# ITEC631 – AI-Driven Network Intrusion Detection System
## Sprint 4: SHAP Explainability

**Student:** Hayoung  
**Unit:** ITEC631 IT Masters Project Part B, Semester 1 2026

Sprint 4 applies **SHAP (SHapley Additive exPlanations)** to the two-stage pipeline built in Sprint 3.  
The goal is to understand *why* the model makes each decision — which features drive BENIGN/ATTACK classification at Stage 1, and which features distinguish attack categories at Stage 2.

**Key question (RQ2):** Which network traffic features are most influential in the NIDS decision-making process, and do Stage 1 and Stage 2 rely on different features?

## Setup

In [ ]:
import os, sys, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
RANDOM_STATE  = 42
SHAP_SAMPLE_S1 = 3000   # samples used for Stage 1 SHAP (speed vs coverage)
SHAP_SAMPLE_S2 = 2000   # samples used for Stage 2 SHAP

print('libraries loaded')
print(f'shap version: {shap.__version__}')

In [ ]:
ON_KAGGLE = os.path.exists('/kaggle/working')

if ON_KAGGLE:
    PROC_DIR     = None
    S2_MODEL_DIR = None
    S3_MODEL_DIR = None

    # locate cicids2017_cleaned.csv (Sprint 1 output)
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'cicids2017_cleaned.csv' in files:
            PROC_DIR = root + '/'
            break

    # locate Sprint 2 models (Stage 1 XGBoost)
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'xgboost.pkl' in files and 'best_model.json' in files:
            S2_MODEL_DIR = root + '/'
            break

    # locate Sprint 3 models (Stage 2 XGBoost + label encoders)
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'stage2_xgboost.pkl' in files:
            S3_MODEL_DIR = root + '/'
            break

    OUT_DIR = '/kaggle/working/'
else:
    PROC_DIR     = 'processed_data/'
    S2_MODEL_DIR = 'models/'
    S3_MODEL_DIR = 'models/'
    OUT_DIR      = ''

print(f'PROC_DIR     : {PROC_DIR}')
print(f'S2_MODEL_DIR : {S2_MODEL_DIR}')
print(f'S3_MODEL_DIR : {S3_MODEL_DIR}')

## 1. Load Data & Models

Reproducing the exact same train/test split used in Sprint 2 and Sprint 3 (stratified 80/20, random_state=42)  
so the test set here is identical to what was evaluated before.

In [ ]:
print('loading cleaned dataset...')
full_df = pd.read_csv(PROC_DIR + 'cicids2017_cleaned.csv')
print(f'loaded {len(full_df):,} rows x {full_df.shape[1]} cols')

feature_cols = [c for c in full_df.columns
                if c not in ('label_binary', 'label_category', 'label_cat_encoded')]
X          = full_df[feature_cols]
y_binary   = full_df['label_binary']
y_category = full_df['label_category']

# Reproduce Sprint 3 split
X_train_raw, X_test, y_bin_train, y_bin_test = train_test_split(
    X, y_binary,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_binary
)
y_cat_test = y_category.loc[y_bin_test.index]

print(f'test set : {X_test.shape[0]:,} rows  |  features : {X_test.shape[1]}')
print(f'test BENIGN : {(y_bin_test==0).sum():,}  |  ATTACK : {(y_bin_test==1).sum():,}')

In [ ]:
# --- Stage 1 model (Sprint 2 best: XGBoost) ---
with open(S2_MODEL_DIR + 'xgboost.pkl', 'rb') as f:
    stage1_model = pickle.load(f)
print('Stage 1 model loaded: XGBoost (binary)')

# --- Stage 2 model (Sprint 3 best: XGBoost) ---
with open(S3_MODEL_DIR + 'stage2_xgboost.pkl', 'rb') as f:
    stage2_model = pickle.load(f)
print('Stage 2 model loaded: XGBoost (multi-class)')

# --- Label encoder for Stage 2 ---
with open(S3_MODEL_DIR + 'stage2_label_encoder.pkl', 'rb') as f:
    le2 = pickle.load(f)

# ensure le2 classes are strings
le2_classes = [str(c) for c in le2.classes_]
# filter out 'nan' class for clean visualisations
valid_classes = [c for c in le2_classes if c != 'nan']
valid_class_idx = [i for i, c in enumerate(le2_classes) if c != 'nan']

print(f'Stage 2 classes: {le2_classes}')

## 2. SHAP — Stage 1: BENIGN vs ATTACK

Using `shap.TreeExplainer` which is optimised for tree-based models (XGBoost, RandomForest).  
It computes exact SHAP values in polynomial time using the tree structure — no approximation needed.

For efficiency, we sample 3,000 records from the test set (representative mix of BENIGN and ATTACK).

In [ ]:
# --- sample for SHAP computation ---
np.random.seed(RANDOM_STATE)
s1_idx = np.random.choice(len(X_test), size=SHAP_SAMPLE_S1, replace=False)
X_shap_s1 = X_test.iloc[s1_idx].reset_index(drop=True)
y_shap_s1 = y_bin_test.iloc[s1_idx].reset_index(drop=True)

print(f'Stage 1 SHAP sample: {len(X_shap_s1):,} records')
print(f'  BENIGN : {(y_shap_s1==0).sum()}  |  ATTACK : {(y_shap_s1==1).sum()}')

# --- compute SHAP values ---
print('computing Stage 1 SHAP values...')
explainer_s1   = shap.TreeExplainer(stage1_model)
shap_values_s1 = explainer_s1.shap_values(X_shap_s1)
# XGBoost binary: shap_values_s1 shape = (n_samples, n_features)
print(f'done  |  shape: {np.array(shap_values_s1).shape}')

In [ ]:
# --- Summary beeswarm plot (Stage 1) ---
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values_s1, X_shap_s1,
    max_display=20,
    show=False,
    plot_type='dot'
)
plt.title('Stage 1 SHAP — Feature Impact on BENIGN vs ATTACK\n(red = pushes toward ATTACK, blue = pushes toward BENIGN)',
          fontsize=11, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(OUT_DIR + 'shap_stage1_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved shap_stage1_beeswarm.png')

In [ ]:
# --- Bar plot: mean |SHAP| (Stage 1) ---
shap_abs_mean_s1 = pd.Series(
    np.abs(shap_values_s1).mean(axis=0),
    index=X_shap_s1.columns
).sort_values(ascending=False)

top20_s1 = shap_abs_mean_s1.head(20).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(top20_s1.index, top20_s1.values, color='#3498db', edgecolor='white', alpha=0.9)
ax.set_title('Stage 1 — Top 20 Features by Mean |SHAP| Value\n(BENIGN vs ATTACK)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Mean |SHAP| Value')
for bar, val in zip(bars, top20_s1.values):
    ax.text(bar.get_width() + 0.0002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR + 'shap_stage1_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved shap_stage1_bar.png')

print('\nTop 10 features for Stage 1 (BENIGN vs ATTACK):')
print(shap_abs_mean_s1.head(10).to_string())

In [ ]:
# --- Waterfall plots: one BENIGN, one ATTACK sample ---
benign_idx = y_shap_s1[y_shap_s1 == 0].index[0]
attack_idx = y_shap_s1[y_shap_s1 == 1].index[0]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, sample_idx, label, color in [
    (axes[0], benign_idx, 'BENIGN', '#2ecc71'),
    (axes[1], attack_idx, 'ATTACK', '#e74c3c')
]:
    top_features = pd.Series(
        shap_values_s1[sample_idx],
        index=X_shap_s1.columns
    ).abs().nlargest(10).index

    vals = pd.Series(shap_values_s1[sample_idx], index=X_shap_s1.columns)[top_features]
    colors = ['#e74c3c' if v > 0 else '#3498db' for v in vals]

    ax.barh(range(len(vals)), vals.values[::-1], color=colors[::-1], edgecolor='white')
    ax.set_yticks(range(len(vals)))
    ax.set_yticklabels(vals.index[::-1], fontsize=9)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Sample Explanation — True Label: {label}\n(top 10 features)',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('SHAP Value\n(positive = toward ATTACK, negative = toward BENIGN)')

plt.suptitle('Stage 1 — Individual Prediction Explanations', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR + 'shap_stage1_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved shap_stage1_waterfall.png')

## 3. SHAP — Stage 2: Attack Category Classification

Stage 2 only processes records flagged as ATTACK by Stage 1.  
Here we compute SHAP values on the **true attack records** from the test set,  
to understand which features help distinguish DoS/DDoS, PortScan, BruteForce, Botnet, and Infiltration.

In [ ]:
# filter attack-only records from test set
attack_mask     = y_bin_test == 1
X_test_attacks  = X_test[attack_mask].reset_index(drop=True)
y_cat_attacks   = y_cat_test[attack_mask].reset_index(drop=True)

print(f'attack records in test set: {len(X_test_attacks):,}')
print(f'category distribution:')
print(y_cat_attacks.value_counts().to_string())

# sample for SHAP (preserve class proportions)
np.random.seed(RANDOM_STATE)
n_s2 = min(SHAP_SAMPLE_S2, len(X_test_attacks))
s2_idx = np.random.choice(len(X_test_attacks), size=n_s2, replace=False)
X_shap_s2 = X_test_attacks.iloc[s2_idx].reset_index(drop=True)
y_shap_s2 = y_cat_attacks.iloc[s2_idx].reset_index(drop=True)

print(f'\nStage 2 SHAP sample: {len(X_shap_s2):,} attack records')

In [ ]:
# compute Stage 2 SHAP values
print('computing Stage 2 SHAP values (multi-class)...')
explainer_s2   = shap.TreeExplainer(stage2_model)
shap_values_s2 = explainer_s2.shap_values(X_shap_s2)

# shap_values_s2 is a list of arrays, one per class
# shape of each: (n_samples, n_features)
if isinstance(shap_values_s2, list):
    n_classes = len(shap_values_s2)
    print(f'done  |  {n_classes} class arrays, each shape: {np.array(shap_values_s2[0]).shape}')
else:
    # newer shap may return (n_samples, n_features, n_classes)
    shap_values_s2 = [shap_values_s2[:, :, i] for i in range(shap_values_s2.shape[2])]
    n_classes = len(shap_values_s2)
    print(f'done  |  converted to list of {n_classes} arrays')

In [ ]:
# --- Summary plot across all attack classes ---
# Stack SHAP values and take mean |SHAP| across all classes
shap_all_classes = np.stack([np.abs(shap_values_s2[i]) for i in valid_class_idx], axis=2)
shap_mean_all    = shap_all_classes.mean(axis=2)  # (n_samples, n_features)

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_mean_all, X_shap_s2,
    max_display=20,
    show=False,
    plot_type='bar',
    color='#e74c3c'
)
plt.title('Stage 2 SHAP — Top 20 Features Averaged Across Attack Classes',
          fontsize=11, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(OUT_DIR + 'shap_stage2_overall.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved shap_stage2_overall.png')

In [ ]:
# --- Per-class top-10 feature bar charts ---
n_valid = len(valid_classes)
fig, axes = plt.subplots(1, n_valid, figsize=(5 * n_valid, 6), sharey=False)
if n_valid == 1:
    axes = [axes]

palette = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

class_top_features = {}

for ax, cls, cls_idx, color in zip(axes, valid_classes, valid_class_idx, palette):
    mean_abs = pd.Series(
        np.abs(shap_values_s2[cls_idx]).mean(axis=0),
        index=X_shap_s2.columns
    ).sort_values(ascending=False)

    top10 = mean_abs.head(10).sort_values()
    class_top_features[cls] = list(mean_abs.head(10).index)

    bars = ax.barh(range(len(top10)), top10.values, color=color, edgecolor='white', alpha=0.85)
    ax.set_yticks(range(len(top10)))
    ax.set_yticklabels(top10.index, fontsize=8)
    ax.set_title(cls, fontsize=11, fontweight='bold')
    ax.set_xlabel('Mean |SHAP|', fontsize=8)

fig.suptitle('Stage 2 — Top 10 Features per Attack Class', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR + 'shap_stage2_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved shap_stage2_per_class.png')

## 4. Cross-Stage Feature Analysis

Comparing which features matter most in Stage 1 vs Stage 2.  
Features that appear in both stages are universally important for the whole pipeline.  
Features unique to Stage 2 are specifically useful for distinguishing attack types.

In [ ]:
# top-20 features for each stage
top20_s1_list = list(shap_abs_mean_s1.head(20).index)
top20_s2_list = list(
    pd.Series(
        shap_mean_all.mean(axis=0),
        index=X_shap_s2.columns
    ).sort_values(ascending=False).head(20).index
)

shared     = set(top20_s1_list) & set(top20_s2_list)
s1_only    = set(top20_s1_list) - set(top20_s2_list)
s2_only    = set(top20_s2_list) - set(top20_s1_list)

print('=== Cross-Stage Feature Overlap ===')
print(f'Top 20 Stage 1 ∩ Top 20 Stage 2 : {len(shared)} shared features')
print(f'  Shared    : {sorted(shared)}')
print(f'  S1 only   : {sorted(s1_only)}')
print(f'  S2 only   : {sorted(s2_only)}')

In [ ]:
# Heatmap: per-class SHAP importance for top-15 features from Stage 2
top15_s2_features = list(
    pd.Series(
        shap_mean_all.mean(axis=0),
        index=X_shap_s2.columns
    ).sort_values(ascending=False).head(15).index
)

heatmap_data = pd.DataFrame(
    index=valid_classes,
    columns=top15_s2_features,
    dtype=float
)

for cls, cls_idx in zip(valid_classes, valid_class_idx):
    mean_abs = pd.Series(
        np.abs(shap_values_s2[cls_idx]).mean(axis=0),
        index=X_shap_s2.columns
    )
    for feat in top15_s2_features:
        heatmap_data.loc[cls, feat] = mean_abs[feat]

# row-normalise so each class sums to 1 (shows relative feature preference)
heatmap_norm = heatmap_data.div(heatmap_data.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    heatmap_norm.astype(float),
    annot=True, fmt='.2f', cmap='YlOrRd',
    linewidths=0.5, cbar_kws={'label': 'Normalised Importance'},
    ax=ax
)
ax.set_title('Stage 2 — Relative Feature Importance per Attack Class\n(row-normalised mean |SHAP|)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Feature', fontsize=10)
ax.set_ylabel('Attack Class', fontsize=10)
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.savefig(OUT_DIR + 'shap_stage2_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved shap_stage2_heatmap.png')

## 5. Save SHAP Results

In [ ]:
# save top features summary as JSON for Sprint 5 dashboard
sprint4_summary = {
    'stage1_top10': list(shap_abs_mean_s1.head(10).index),
    'stage1_top10_scores': list(shap_abs_mean_s1.head(10).values.round(6)),
    'stage2_top10_overall': top20_s2_list[:10],
    'stage2_per_class': class_top_features,
    'cross_stage_shared': sorted(list(shared)),
    'shap_sample_s1': SHAP_SAMPLE_S1,
    'shap_sample_s2': SHAP_SAMPLE_S2,
}

with open(OUT_DIR + 'sprint4_shap_summary.json', 'w') as f:
    json.dump(sprint4_summary, f, indent=2)
print(f'saved {OUT_DIR}sprint4_shap_summary.json')

# save stage1 SHAP values as numpy array
np.save(OUT_DIR + 'shap_values_stage1.npy', shap_values_s1)
print(f'saved shap_values_stage1.npy')

## 6. Sprint 4 Summary

In [ ]:
print('Sprint 4 — SHAP Explainability Complete')
print('=' * 60)
print()
print('Stage 1 (BENIGN vs ATTACK) — Top 5 features:')
for i, (feat, val) in enumerate(shap_abs_mean_s1.head(5).items(), 1):
    print(f'  {i}. {feat:<40} {val:.4f}')
print()
print('Stage 2 (Attack Category) — Top 5 features (overall):')
stage2_overall = pd.Series(
    shap_mean_all.mean(axis=0),
    index=X_shap_s2.columns
).sort_values(ascending=False)
for i, (feat, val) in enumerate(stage2_overall.head(5).items(), 1):
    print(f'  {i}. {feat:<40} {val:.4f}')
print()
print(f'Cross-stage shared top features ({len(shared)}): {sorted(shared)}')
print()
print('Outputs saved:')
for fname in ['shap_stage1_beeswarm.png', 'shap_stage1_bar.png',
              'shap_stage1_waterfall.png', 'shap_stage2_overall.png',
              'shap_stage2_per_class.png', 'shap_stage2_heatmap.png',
              'sprint4_shap_summary.json', 'shap_values_stage1.npy']:
    print(f'  {OUT_DIR}{fname}')
print()
print('Next: Sprint 5 — Streamlit dashboard')
print('      Visualise pipeline decisions and SHAP explanations interactively')